# Extraction d'embeddings avec BEATs

Ce notebook extrait des embeddings audio avec le modèle BEATs (Microsoft).

## Étapes :
1. Charge un fichier ZIP contenant des fichiers audio (.wav)
2. Rééchantillonne chaque audio de 8 kHz vers 16 kHz
3. Calcule les embeddings avec le modèle pré-entraîné BEATs
4. Sauvegarde les embeddings dans un fichier .npy

## Cell 1: Google Colab Setup & Google Drive Mount

In [1]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully")
else:
    print("Running locally (not in Google Colab)")

# Set the path to your CNRS project
if IN_COLAB:
    PROJECT_PATH = '/content/drive/MyDrive/audio_simon_moutier'
else:
    PROJECT_PATH = 'C:/Users/Simon/Desktop/CNRS'

print(f"Project path: {PROJECT_PATH}")

Mounted at /content/drive
Google Drive mounted successfully
Project path: /content/drive/MyDrive/audio_simon_moutier


## Cell 2: Install Dependencies

In [2]:
if IN_COLAB:
    print("Installing required packages")
    !pip install -q soundfile librosa torch tqdm pandas pyarrow
    print("All packages installed")

Installing required packages
All packages installed


## Cell 3: Clone BEATs Repository

In [3]:
import os

# Path to BEATs repo
BEATS_DIR = os.path.join(PROJECT_PATH, 'Scripts/Embedding_Extraction/embed_extraction_BEATs/beats')

# Add to path
sys.path.insert(0, BEATS_DIR)

## Cell 4: Imports

In [4]:
import os
import zipfile
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
from tqdm import tqdm

from BEATs import BEATs, BEATsConfig

print("All imports successful")

All imports successful


## Cell 5: Configuration

In [5]:
# Paths
ZIP_PATH = os.path.join(PROJECT_PATH, 'Data/Audio/AudioRaw/3s_samples_subset.zip')

CHECKPOINT_PATH = os.path.join(
    PROJECT_PATH,
    'Scripts/Embedding_Extraction/embed_extraction_BEATs/BEATs_iter3_plus_AS2M.pt'
)

OUTPUT_DIR = os.path.join(PROJECT_PATH, 'embeddings')
OUTPUT_PARQUET = os.path.join(OUTPUT_DIR, 'beats_embeddings.parquet')

# Settings
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TARGET_SR = 16000

print(f"Device: {DEVICE}")
print(f"ZIP path: {ZIP_PATH}")
print(f"Checkpoint: {CHECKPOINT_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Output parquet file: {OUTPUT_PARQUET}")

# Verify paths
if not os.path.exists(ZIP_PATH):
    print(f"WARNING: ZIP file not found at {ZIP_PATH}")
if not os.path.exists(CHECKPOINT_PATH):
    print(f"WARNING: Checkpoint not found at {CHECKPOINT_PATH}")

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

Device: cuda
ZIP path: /content/drive/MyDrive/audio_simon_moutier/Data/Audio/AudioRaw/3s_samples_subset.zip
Checkpoint: /content/drive/MyDrive/audio_simon_moutier/Scripts/Embedding_Extraction/embed_extraction_BEATs/BEATs_iter3_plus_AS2M.pt
Output directory: /content/drive/MyDrive/audio_simon_moutier/embeddings
Output parquet file: /content/drive/MyDrive/audio_simon_moutier/embeddings/beats_embeddings.parquet


## Cell 6: Load BEATs Model

In [6]:
print("Loading BEATs model")
print(f"Using device: {DEVICE}")

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

cfg = BEATsConfig(checkpoint["cfg"])

model = BEATs(cfg)
model.load_state_dict(checkpoint["model"])

model.eval()
model.to(DEVICE)

print("Model loaded successfully")

Loading BEATs model
Using device: cuda


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Model loaded successfully


## Cell 7: Define Embedding Extraction Function

In [7]:
@torch.no_grad()
def extract_embedding(audio_path):
    """
    Extract BEATs embedding from audio file.

    Parameters:
    -----------
    audio_path : str
        Path to the audio file (.wav)

    Returns:
    --------
    embedding : np.ndarray
        1D embedding vector of shape (768,)
    """

    # --------------------------------------------------------
    # Read audio
    # --------------------------------------------------------
    waveform, sr = sf.read(audio_path)

    # Stereo -> Mono
    if len(waveform.shape) > 1:
        waveform = np.mean(waveform, axis=1)

    # --------------------------------------------------------
    # Resample 8 kHz -> 16 kHz
    # --------------------------------------------------------
    if sr != TARGET_SR:
        waveform = librosa.resample(
            waveform,
            orig_sr=sr,
            target_sr=TARGET_SR
        )

    # --------------------------------------------------------
    # Convert to tensor
    # --------------------------------------------------------
    waveform = torch.tensor(waveform).float().unsqueeze(0).to(DEVICE)

    # --------------------------------------------------------
    # Extract features
    # --------------------------------------------------------
    # Create padding mask - all zeros means all positions are valid
    # Use proper float dtype for better compatibility
    padding_mask = torch.zeros(
        waveform.shape,
        dtype=torch.bool,
        device=DEVICE
    )

    features = model.extract_features(
        waveform,
        padding_mask=padding_mask
    )[0]

    # features shape:
    # [batch, time, dim]

    embedding = features.mean(dim=1)

    return embedding.squeeze(0).cpu().numpy()


print("Embedding extraction function defined")

Embedding extraction function defined


## Cell 8: Extract Embeddings from ZIP

In [8]:
all_embeddings = []
all_filenames = []
error_count = 0

with tempfile.TemporaryDirectory() as tmpdir:

    print(f"Extracting ZIP file...")

    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(tmpdir)

    audio_files = list(Path(tmpdir).rglob("*.wav"))

    print(f"Found {len(audio_files)} audio files")
    print()

    for audio_path in tqdm(audio_files, desc="Extracting embeddings"):

        try:
            emb = extract_embedding(str(audio_path))

            all_embeddings.append(emb)
            all_filenames.append(audio_path.name)

        except Exception as e:
            error_count += 1
            if error_count <= 5:  # Print first 5 errors only
                print(f"\nError with {audio_path.name}: {str(e)[:100]}")

print(f"\n{'='*60}")
print(f"Successfully extracted: {len(all_embeddings)} files")
print(f"Failed: {error_count} files")
print(f"Success rate: {len(all_embeddings) / len(audio_files) * 100:.1f}%")
print(f"{'='*60}")

Extracting ZIP file...
Found 10325 audio files



Extracting embeddings: 100%|██████████| 10325/10325 [04:38<00:00, 37.13it/s]



Successfully extracted: 10325 files
Failed: 0 files
Success rate: 100.0%


## Cell 9: Save Embeddings

In [9]:
all_embeddings = np.stack(all_embeddings)

# Create a DataFrame with embeddings and filenames
# Each embedding (768 dims) becomes a column
embedding_df = pd.DataFrame(
    all_embeddings,
    columns=[f"embedding_{i}" for i in range(all_embeddings.shape[1])]
)

# Add filenames column at the beginning
embedding_df.insert(0, 'filename', all_filenames)

# Save to Parquet
embedding_df.to_parquet(OUTPUT_PARQUET, index=False)

print("Embeddings saved successfully.")
print(f"Embeddings shape: {all_embeddings.shape}")
print(f"DataFrame shape: {embedding_df.shape}")
print(f"Saved to: {OUTPUT_PARQUET}")

Embeddings saved successfully.
Embeddings shape: (10325, 768)
DataFrame shape: (10325, 769)
Saved to: /content/drive/MyDrive/audio_simon_moutier/embeddings/beats_embeddings.parquet


## Cell 10: Summary Statistics

In [10]:
print("\n" + "="*60)
print("EXTRACTION COMPLETE")
print("="*60)
print(f"Total audio files processed: {len(all_embeddings)}")
print(f"Embedding dimension: {all_embeddings.shape[1]}")
print(f"Device used: {DEVICE}")
print(f"Target sample rate: {TARGET_SR} Hz")
print()
print("Output file:")
print(f"  - {OUTPUT_PARQUET}")
print(f"\nFile format: Parquet (contains filenames + embeddings)")
print("="*60)


EXTRACTION COMPLETE
Total audio files processed: 10325
Embedding dimension: 768
Device used: cuda
Target sample rate: 16000 Hz

Output file:
  - /content/drive/MyDrive/audio_simon_moutier/embeddings/beats_embeddings.parquet

File format: Parquet (contains filenames + embeddings)
